# Master orchestrator (OSS)

Drive the local medallion demo: load manufacturing CSVs from `data/raw/`, then run Bronze → Silver → Gold helpers from `lib/ambient_pipeline/`.

No Databricks, Unity Catalog, or Firebase required.


In [ ]:
from pathlib import Path
from ambient_pipeline.notebook_bootstrap import ensure_pipeline_on_path, apply_spark_tuning

ROOT = ensure_pipeline_on_path(Path.cwd())
assert ROOT is not None, "Run from ambient-core checkout (lib/ambient_pipeline missing)"
print(f"repo root: {ROOT}")


In [ ]:
from ambient_pipeline.perf import create_local_spark

spark = create_local_spark(app_name="ambient-oss-notebooks", shuffle_partitions=4)
apply_spark_tuning(spark)
print(spark.version)


In [ ]:
from pathlib import Path

raw_dir = ROOT / "data" / "raw"
csv_files = sorted(raw_dir.glob("Allmanufacturingds-*.csv"))
assert csv_files, f"No demo CSVs in {raw_dir}"
dfs = {}
for path in csv_files:
    key = path.stem
    dfs[key] = spark.read.option("header", True).csv(str(path))
    print(f"loaded {key}: rows={dfs[key].count()} cols={len(dfs[key].columns)}")
print(f"total tables: {len(dfs)}")


## Run stages

Execute the sibling notebooks in order, or call the headless smoke script:

```bash
python scripts/run_oss_bronze_silver_smoke.py
```

Notebook order: `01_bronze_ingestion` → `02_silver_transformation` → `03_gold_metrics` → `04_local_delta_governance`.


In [ ]:
# Optional: invoke the OSS smoke path from this kernel
import subprocess, sys
result = subprocess.run([sys.executable, str(ROOT / "scripts" / "run_oss_bronze_silver_smoke.py")], cwd=str(ROOT))
print("smoke exit:", result.returncode)
assert result.returncode == 0
